# Two-Bunch BOED — Real Experiment

Amortized BOED generator for T0 search using gunphase scans.

Protocol:
1. Grid scan: `grid_steps` evenly-spaced gunphase measurements via the hardware evaluator.
2. BOED loop: posterior-sampled designs from traced round models. Normalization is fitted automatically from the grid scan on the first BOED call.
3. Diagnostic plot: measurements + T0 posterior + posterior predictive.

Update `model_dir`, `design_range`, and the `_evaluate` function before running.

In [ ]:
# ── Config — update for each run ──────────────────────────────────────────────
model_dir = "aboed"  # directory with traced .pt files
design_range = [-25.0, 45.0]  # [t_min, t_max] gunphase in degrees
observable_name = "charge"
variable_name = "gunphase"
max_measure = 100  # total measurements (grid + BOED)

n_posterior_samples = 2000
n_predictive_curves = 300

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from xopt import Evaluator, VOCS, Xopt

from autonomous_control.facet.two_bunch_boed_utils import (
    TwoBunchDoubleExp,
    denormalize_nuisance,
    find_full_param_model,
    compute_predictive_curves,
    AmortizedBOEDBunchGenerator,
)

In [ ]:
def _evaluate(x):
    # TODO: need to implement this function to call the actual measurement hardware.
    return {observable_name: 0.0}

In [ ]:
# ── Xopt setup ────────────────────────────────────────────────────────────────

vocs = VOCS(
    variables={variable_name: design_range},
    observables=[observable_name],
)
generator = AmortizedBOEDBunchGenerator(
    model_dir=model_dir,
    design_range=design_range,
    observable_name=observable_name,
    vocs=vocs,
)
# Extract the number of grid steps from the generator for use in the experiment loop.
grid_steps = generator.grid_steps
evaluator = Evaluator(function=_evaluate)
X = Xopt(vocs=vocs, generator=generator, evaluator=evaluator)

In [ ]:
# ── Experiment loop (grid scan + BOED) ────────────────────────────────────────
# The generator handles the grid phase internally for the first `grid_steps`
# calls, then auto-fits normalization and switches to BOED.

print(
    f"Running {max_measure} steps ({grid_steps} grid + {max_measure - grid_steps} BOED)..."
)

for step in range(max_measure):
    X.step()

    t_vals = X.data[variable_name].values
    y_vals = X.data[observable_name].values
    n_obs = len(t_vals)
    order = np.arange(n_obs)

    # After grid scan completes, show grid overview once
    if n_obs == grid_steps:
        plt.figure(figsize=(8, 3))
        plt.scatter(t_vals, y_vals, c="steelblue", s=40, zorder=3)
        plt.xlabel("gunphase (°)")
        plt.ylabel(observable_name)
        plt.title("Grid scan")
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        continue

    # Still in grid phase — no per-step plot
    if n_obs < grid_steps:
        continue

    # BOED per-step diagnostic
    boed_step = n_obs - grid_steps
    print(f"BOED step {boed_step} / {max_measure - grid_steps}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax = axes[0]
    ax.scatter(
        t_vals[:grid_steps],
        y_vals[:grid_steps],
        c="steelblue",
        s=30,
        zorder=4,
        label="grid",
    )
    sc = ax.scatter(
        t_vals[grid_steps:],
        y_vals[grid_steps:],
        c=order[grid_steps:],
        cmap="YlOrRd",
        s=30,
        zorder=5,
        label="BOED",
    )
    plt.colorbar(sc, ax=ax, label="BOED step")
    ax.set_xlabel("gunphase (°)")
    ax.set_ylabel(observable_name)
    ax.set_title("Signal vs gunphase")
    ax.legend(fontsize=8)
    ax.grid(True)

    ax = axes[1]
    ax.plot(order, t_vals, "o-", ms=4)
    ax.set_xlabel("step")
    ax.set_ylabel("gunphase (°)")
    ax.set_title("Sampling sequence")
    ax.grid(True)

    plt.tight_layout()
    plt.show()

print("Done.")

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
out_csv = "two_bunch_boed_run.csv"
X.data.to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")

In [ ]:
# ── Final diagnostic plot ─────────────────────────────────────────────────────

t_vals = X.data[variable_name].values
y_vals = X.data[observable_name].values
order = np.arange(len(t_vals))
norm = X.generator._normalizer

# Normalized history for model input
xi_norm_all = norm.normalize_x(
    torch.tensor(t_vals, dtype=torch.float32).unsqueeze(1)
).unsqueeze(0)
y_norm_all = norm.normalize_y(
    torch.tensor(y_vals, dtype=torch.float32).unsqueeze(1)
).unsqueeze(0)

# T0 posterior samples from the last round model
last_round = max(X.generator._available_rounds)
with torch.no_grad():
    means, stds, log_weights = generator._load_model(last_round)(
        xi_norm_all, y_norm_all
    )
weights = torch.softmax(log_weights.squeeze(0), dim=-1)
idx = torch.multinomial(weights, n_posterior_samples, replacement=True)
samples_norm = (
    means[0, idx, :]
    + stds[0, idx, :] * torch.randn(n_posterior_samples, means.shape[-1])
)[:, 0].clamp(0.0, 1.0)
t0_samples = norm.denormalize_t0(samples_norm).numpy()

fig, axes = plt.subplots(3, 1, figsize=(10, 12))

# ── Panel 1: measurements ─────────────────────────────────────────────────────
ax = axes[0]
ax.scatter(
    t_vals[:grid_steps],
    y_vals[:grid_steps],
    c="steelblue",
    s=40,
    zorder=4,
    label=f"grid (n={grid_steps})",
)
if len(t_vals) > grid_steps:
    sc = ax.scatter(
        t_vals[grid_steps:],
        y_vals[grid_steps:],
        c=order[grid_steps:],
        cmap="YlOrRd",
        s=40,
        zorder=5,
        label="BOED",
    )
    plt.colorbar(sc, ax=ax, label="BOED step")
ax.set_xlabel("gunphase (°)")
ax.set_ylabel(observable_name)
ax.set_title("Measurements")
ax.legend(fontsize=8)
ax.grid(True)

# ── Panel 2: T0 posterior ─────────────────────────────────────────────────────
ax = axes[1]
med = float(np.median(t0_samples))
lo68, hi68 = float(np.percentile(t0_samples, 16)), float(np.percentile(t0_samples, 84))
ax.hist(t0_samples, bins=60, density=True, color="steelblue", alpha=0.75)
ax.axvline(med, color="k", lw=1.2, label=f"median={med:.2f}°")
ax.axvspan(lo68, hi68, alpha=0.15, color="k", label=f"68% [{lo68:.1f}, {hi68:.1f}]°")
ax.set_xlabel("T0 (°)")
ax.set_ylabel("density")
ax.set_title(f"T0 posterior  (round {last_round} model, n_obs={len(t_vals)})")
ax.legend(fontsize=8)
ax.grid(True)

# ── Panel 3: posterior predictive ─────────────────────────────────────────────
ax = axes[2]
full_path = find_full_param_model(model_dir)
if full_path is not None:
    full_model = torch.jit.load(str(full_path), map_location="cpu")
    full_model.eval()
    with torch.no_grad():
        m_f, s_f, lw_f = full_model(xi_norm_all, y_norm_all)
    w_f = torch.softmax(lw_f.squeeze(0), dim=-1)
    idx_f = torch.multinomial(w_f, n_predictive_curves, replacement=True)
    samp_f = (
        m_f[0, idx_f, :]
        + s_f[0, idx_f, :] * torch.randn(n_predictive_curves, m_f.shape[-1])
    ).clamp(0.0, 1.0)

    t0_phys_f = norm.denormalize_t0(samp_f[:, 0]).numpy()
    nu_phys_f = denormalize_nuisance(samp_f[:, 1:]).numpy()

    t_fine = np.linspace(t_vals.min() - 2, t_vals.max() + 2, 400)
    t_fine_t = torch.tensor(t_fine, dtype=torch.float32).unsqueeze(1)
    t_grid_t = torch.tensor(t_vals[:grid_steps], dtype=torch.float32).unsqueeze(1)
    sim = TwoBunchDoubleExp()
    curves_norm = compute_predictive_curves(
        t0_phys_f, nu_phys_f, t_fine_t, t_grid_t, sim, n_predictive_curves
    )

    p025, p16, p50, p84, p975 = [
        np.percentile(curves_norm, q, axis=0) for q in (2.5, 16, 50, 84, 97.5)
    ]
    ax.fill_between(t_fine, p025, p975, color="tomato", alpha=0.20, label="±2σ")
    ax.fill_between(t_fine, p16, p84, color="tomato", alpha=0.40, label="±1σ")
    ax.plot(t_fine, p50, color="tomato", lw=1.5, label="median")

    y_norm_obs = norm.normalize_y(torch.tensor(y_vals)).numpy()
    ax.scatter(
        t_vals[:grid_steps],
        y_norm_obs[:grid_steps],
        c="steelblue",
        s=30,
        zorder=5,
        label="grid",
    )
    if len(t_vals) > grid_steps:
        ax.scatter(
            t_vals[grid_steps:],
            y_norm_obs[grid_steps:],
            c="tomato",
            s=30,
            zorder=5,
            label="BOED",
        )
    ax.set_xlabel("gunphase (°)")
    ax.set_ylabel("signal (normalized)")
    ax.set_title("Posterior predictive")
    ax.legend(fontsize=8)
    ax.grid(True)
else:
    ax.text(
        0.5,
        0.5,
        "No full_param_posterior_traced_horizon_*.pt found in model_dir",
        ha="center",
        va="center",
        transform=ax.transAxes,
        fontsize=10,
    )

fig.suptitle(f"Two-bunch BOED  (n_obs={len(t_vals)})", fontsize=12)
plt.tight_layout()
plt.show()